# SFT Fine-Tuning: Qwen3-1.7B for STEM Tutoring

This notebook fine-tunes **Qwen3-1.7B** using Supervised Fine-Tuning (SFT) with QLoRA
for multi-domain STEM tutoring in Socratic style.

**Training pipeline stage:** 1 of 4 (SFT -> SimPO -> GRPO -> STaR)

**Key advantage:** This smaller model can also run **locally** on consumer GPUs (e.g., RTX 2080 with 8GB VRAM).

| Environment | Batch Size | Seq Length | Notes |
|-------------|-----------|------------|-------|
| Colab A100  | 8         | 2048       | Full speed |
| RTX 2080 (8GB) | 1      | 1024       | Local training possible |
| RTX 3090 (24GB) | 4     | 2048       | Good local option |

**Domains:** Mathematics, Physics, Chemistry, Biology, Computer Science

In [ ]:
# Install dependencies
!pip install -q unsloth trl peft transformers datasets
!pip install -q accelerate bitsandbytes sentencepiece protobuf

In [ ]:
# ============================================================
# Configuration
# ============================================================
import torch

# Model
MODEL = "unsloth/Qwen3-1.7B"

# Auto-detect environment and set batch size / seq length accordingly
if torch.cuda.is_available():
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_mem / (1024 ** 3)
    GPU_NAME = torch.cuda.get_device_name(0)
    print(f"GPU: {GPU_NAME} ({gpu_mem_gb:.1f} GB)")

    if gpu_mem_gb >= 35:  # A100 / A6000
        BATCH_SIZE = 8
        MAX_SEQ = 2048
        ENV_NAME = "A100"
    elif gpu_mem_gb >= 20:  # RTX 3090 / 4090
        BATCH_SIZE = 4
        MAX_SEQ = 2048
        ENV_NAME = "RTX 3090/4090"
    else:  # RTX 2080 / consumer GPU
        BATCH_SIZE = 1
        MAX_SEQ = 1024
        ENV_NAME = "RTX 2080 (low VRAM)"
else:
    raise RuntimeError("No GPU detected. GPU is required for training.")

print(f"Environment: {ENV_NAME} -> batch_size={BATCH_SIZE}, max_seq={MAX_SEQ}")

# LoRA hyperparameters
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
TARGET_MODULES = "all-linear"

# Quantization
LOAD_IN_4BIT = True
BNB_4BIT_QUANT_TYPE = "nf4"

# Training hyperparameters
EPOCHS = 3
LEARNING_RATE = 2e-4
GRADIENT_ACCUMULATION_STEPS = max(1, 16 // BATCH_SIZE)  # keep effective batch ~16
LR_SCHEDULER = "cosine"
WARMUP_STEPS = 50
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

# Checkpointing
SAVE_STEPS = 500
LOGGING_STEPS = 10

# Paths
DATASET_PATH = "training/data/combined_stem_balanced.jsonl"
OUTPUT_DIR = "/content/drive/MyDrive/MITS/checkpoints/sft_qwen3_1.7b"
VALIDATION_SPLIT = 0.05

# For local training, override output dir
IS_COLAB = False
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    OUTPUT_DIR = "./checkpoints/sft_qwen3_1.7b"
    print(f"Local mode: saving to {OUTPUT_DIR}")

# Domains for balanced sampling
DOMAINS = ["math", "physics", "chemistry", "biology", "cs"]

# Difficulty mapping (Russian labels → curriculum categories)
EASY_DIFFICULTIES = {"школьный"}
MEDIUM_DIFFICULTIES = {"базовый университетский"}
HARD_DIFFICULTIES = {"продвинутый", "олимпиадный"}

print(f"\nModel: {MODEL}")
print(f"LoRA rank: {LORA_R}, alpha: {LORA_ALPHA}")
print(f"Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Epochs: {EPOCHS}, LR: {LEARNING_RATE}")
print(f"Dataset: {DATASET_PATH}")

In [ ]:
# ============================================================
# Mount Google Drive (optional) and load dataset from HuggingFace
# ============================================================
import json
import os
from pathlib import Path
from collections import Counter

# Try mounting Drive for checkpoint persistence; fall back to local
DRIVE_MOUNTED = False
if IS_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_MOUNTED = True
        print("Google Drive mounted successfully")
    except Exception as e:
        print(f"Drive mount failed ({e}), using local storage")
        OUTPUT_DIR = "/content/checkpoints/sft_qwen3_1.7b"

# Load from HuggingFace Hub
from datasets import load_dataset

print("Loading SFT dataset from Siesher/mits-stem-training-data...")
hf_ds = load_dataset("Siesher/mits-stem-training-data", "sft")

train_records = [dict(r) for r in hf_ds["train"]]
val_records = [dict(r) for r in hf_ds["test"]]

print(f"Train: {len(train_records)}, Validation: {len(val_records)}")

# Analyze domain distribution
domain_counts = Counter(r.get("domain", "unknown") for r in train_records)
for domain, count in sorted(domain_counts.items()):
    print(f"  {domain}: {count} examples")

# Analyze difficulty distribution
difficulty_counts = Counter(r.get("difficulty", "unknown") for r in train_records)
for diff, count in sorted(difficulty_counts.items()):
    print(f"  {diff}: {count} examples")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\nCheckpoints will be saved to: {OUTPUT_DIR}")

In [ ]:
# ============================================================
# Load model with 4-bit quantization and apply LoRA
# ============================================================
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_SEQ,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,  # auto-detect
)

print(f"Model loaded: {MODEL}")
print(f"Model dtype: {model.dtype}")
print(f"Tokenizer vocab size: {len(tokenizer)}")

# Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTrainable parameters: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")

In [ ]:
# ============================================================
# Domain-balanced sampler
# ============================================================
import torch
from torch.utils.data import Sampler
from collections import defaultdict
import math


class DomainBalancedSampler(Sampler):
    """Sampler that ensures each batch contains examples from all 5 STEM domains.

    For each batch, we pick ceil(batch_size / num_domains) examples from each domain
    in a round-robin fashion, ensuring balanced domain exposure during training.
    """

    def __init__(self, dataset_records, batch_size, domains=None, seed=42):
        self.batch_size = batch_size
        self.domains = domains or DOMAINS
        self.seed = seed

        # Group indices by domain
        self.domain_indices = defaultdict(list)
        for idx, record in enumerate(dataset_records):
            domain = record.get("domain", "unknown")
            if domain in self.domains:
                self.domain_indices[domain].append(idx)
            else:
                self.domain_indices[self.domains[idx % len(self.domains)]].append(idx)

        self.num_samples = sum(len(v) for v in self.domain_indices.values())
        self.per_domain_per_batch = max(1, self.batch_size // len(self.domains))

        print(f"DomainBalancedSampler: {self.num_samples} samples across {len(self.domains)} domains")
        for d in self.domains:
            print(f"  {d}: {len(self.domain_indices.get(d, []))} samples")

    def __iter__(self):
        rng = torch.Generator()
        rng.manual_seed(self.seed)

        shuffled = {}
        for domain in self.domains:
            indices = self.domain_indices[domain].copy()
            perm = torch.randperm(len(indices), generator=rng).tolist()
            shuffled[domain] = [indices[i] for i in perm]

        domain_pointers = {d: 0 for d in self.domains}
        all_indices = []

        total_batches = self.num_samples // self.batch_size
        for _ in range(total_batches):
            batch = []
            for domain in self.domains:
                for _ in range(self.per_domain_per_batch):
                    if domain_pointers[domain] >= len(shuffled[domain]):
                        domain_pointers[domain] = 0
                    batch.append(shuffled[domain][domain_pointers[domain]])
                    domain_pointers[domain] += 1
            batch = batch[:self.batch_size]
            perm = torch.randperm(len(batch), generator=rng).tolist()
            all_indices.extend([batch[i] for i in perm])

        return iter(all_indices)

    def __len__(self):
        return (self.num_samples // self.batch_size) * self.batch_size

In [ ]:
# ============================================================
# Curriculum dataset: progressive difficulty across epochs
# ============================================================
from torch.utils.data import Dataset


class CurriculumSTEMDataset(Dataset):
    """Dataset that implements curriculum learning across epochs.

    - Epoch 1: Only easy + medium examples (build foundation)
    - Epoch 2: All examples (easy + medium + hard)
    - Epoch 3: All examples, but hard examples are weighted 2x
    """

    def __init__(self, records, tokenizer, max_seq_length=2048):
        self.all_records = records
        self.tokenizer = tokenizer
        self.max_seq_length = max_seq_length
        self.current_epoch = 1

        # Pre-categorize by difficulty using Russian labels
        self.easy_medium = [
            r for r in records
            if r.get("difficulty", "") in EASY_DIFFICULTIES | MEDIUM_DIFFICULTIES
        ]
        self.hard = [
            r for r in records
            if r.get("difficulty", "") in HARD_DIFFICULTIES
        ]

        # Fallback: if no records matched, treat all as easy_medium
        if not self.easy_medium and not self.hard:
            print("WARNING: No difficulty labels matched. Using all records as easy/medium.")
            self.easy_medium = records.copy()
            self.hard = []

        print(f"CurriculumSTEMDataset: {len(self.easy_medium)} easy/medium, {len(self.hard)} hard")
        self._build_epoch_data()

    def set_epoch(self, epoch):
        self.current_epoch = epoch
        self._build_epoch_data()
        print(f"Epoch {epoch}: {len(self.active_records)} active examples")

    def _build_epoch_data(self):
        if self.current_epoch == 1:
            self.active_records = self.easy_medium.copy()
        elif self.current_epoch == 2:
            self.active_records = self.all_records.copy()
        else:
            self.active_records = self.easy_medium.copy() + self.hard * 2

    def __len__(self):
        return len(self.active_records)

    def __getitem__(self, idx):
        record = self.active_records[idx]

        messages = record.get("messages", [])
        if not messages:
            system_msg = "Ты — сократический репетитор по математике, физике, химии, информатике и биологии. Помогай студентам, задавая наводящие вопросы."
            messages = [
                {"role": "system", "content": system_msg},
                {"role": "user", "content": record.get("instruction", record.get("input", ""))},
                {"role": "assistant", "content": record.get("output", record.get("response", ""))},
            ]

        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

        return {"text": text}


train_dataset = CurriculumSTEMDataset(train_records, tokenizer, MAX_SEQ)
val_dataset = CurriculumSTEMDataset(val_records, tokenizer, MAX_SEQ)
val_dataset.set_epoch(2)  # validation always uses all difficulties

In [ ]:
# ============================================================
# SFTTrainer setup and training with curriculum + checkpointing
# ============================================================
from trl import SFTTrainer
from transformers import TrainingArguments

os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,  # We control epochs manually for curriculum
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    seed=42,
    report_to="none",
    dataloader_pin_memory=True,
    remove_unused_columns=False,
)

# Training loop across curriculum epochs
all_logs = []

for epoch in range(1, EPOCHS + 1):
    print(f"\n{'='*60}")
    print(f"CURRICULUM EPOCH {epoch}/{EPOCHS}")
    print(f"{'='*60}")

    train_dataset.set_epoch(epoch)

    if epoch > 1:
        training_args.learning_rate = LEARNING_RATE * (0.5 ** (epoch - 1))
        training_args.warmup_steps = 0
        print(f"  LR adjusted to {training_args.learning_rate}")

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        max_seq_length=MAX_SEQ,
        dataset_text_field="text",
        packing=True,
        args=training_args,
    )

    resume_ckpt = None
    if epoch == 1 and os.path.exists(os.path.join(OUTPUT_DIR, "checkpoint-latest")):
        resume_ckpt = os.path.join(OUTPUT_DIR, "checkpoint-latest")
        print(f"  Resuming from {resume_ckpt}")

    result = trainer.train(resume_from_checkpoint=resume_ckpt)
    all_logs.append({
        "epoch": epoch,
        "train_loss": result.training_loss,
        "metrics": result.metrics,
    })

    epoch_dir = os.path.join(OUTPUT_DIR, f"epoch-{epoch}")
    trainer.save_model(epoch_dir)
    print(f"  Saved checkpoint to {epoch_dir}")

print("\nTraining complete!")
for log in all_logs:
    print(f"  Epoch {log['epoch']}: loss={log['train_loss']:.4f}")

In [ ]:
# ============================================================
# Evaluate on validation split
# ============================================================
from collections import defaultdict
import math

print("Evaluating on held-out validation set...")

eval_results = trainer.evaluate()
print(f"\nOverall eval loss: {eval_results.get('eval_loss', 'N/A'):.4f}")
print(f"Eval perplexity: {math.exp(eval_results.get('eval_loss', 0)):.2f}")

# Per-domain evaluation
FastLanguageModel.for_inference(model)

domain_metrics = defaultdict(lambda: {"correct": 0, "total": 0})

for record in val_records[:200]:
    domain = record.get("domain", "unknown")
    messages = record.get("messages", [])
    if not messages:
        continue

    input_messages = [m for m in messages if m["role"] != "assistant"]
    prompt = tokenizer.apply_chat_template(
        input_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            do_sample=True,
        )

    generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    domain_metrics[domain]["total"] += 1

    if "?" in generated:
        domain_metrics[domain]["correct"] += 1

print("\nPer-domain Socratic style adherence:")
for domain in DOMAINS:
    m = domain_metrics[domain]
    if m["total"] > 0:
        pct = 100 * m["correct"] / m["total"]
        print(f"  {domain}: {pct:.1f}% ({m['correct']}/{m['total']})")

metrics_path = os.path.join(OUTPUT_DIR, "eval_metrics.json")
with open(metrics_path, "w") as f:
    json.dump({
        "eval_results": eval_results,
        "domain_metrics": dict(domain_metrics),
        "curriculum_logs": all_logs,
        "environment": ENV_NAME,
    }, f, indent=2, default=str)
print(f"\nMetrics saved to {metrics_path}")

In [ ]:
# ============================================================
# Save LoRA adapter to Drive + optional HuggingFace push
# ============================================================

final_adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
model.save_pretrained(final_adapter_path)
tokenizer.save_pretrained(final_adapter_path)
print(f"Final LoRA adapter saved to {final_adapter_path}")

# Save training config for reproducibility
config_to_save = {
    "model": MODEL,
    "max_seq_length": MAX_SEQ,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "target_modules": TARGET_MODULES,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "scheduler": LR_SCHEDULER,
    "warmup_steps": WARMUP_STEPS,
    "quantization": "4bit_nf4",
    "environment": ENV_NAME,
    "train_samples": len(train_records),
    "val_samples": len(val_records),
}
config_path = os.path.join(OUTPUT_DIR, "training_config.json")
with open(config_path, "w") as f:
    json.dump(config_to_save, f, indent=2)
print(f"Config saved to {config_path}")

# Optional: Push to HuggingFace Hub
PUSH_TO_HUB = False
HF_REPO_ID = "your-username/mits-qwen3-1.7b-sft"

if PUSH_TO_HUB:
    from huggingface_hub import login
    login()

    model.push_to_hub(HF_REPO_ID, private=True)
    tokenizer.push_to_hub(HF_REPO_ID, private=True)
    print(f"Pushed to https://huggingface.co/{HF_REPO_ID}")
else:
    print("Skipping HuggingFace push (set PUSH_TO_HUB=True to enable)")

print("\nDone! Adapter is ready for SimPO alignment (next stage).")